# Planning


#### Python Packages to Install

In [ ]:
# Core
fastapi==0.115.0
uvicorn==0.30.0
sqlalchemy==2.0.0
psycopg2-binary==2.9.0

# AI/ML
sentence-transformers==3.0.0
faiss-cpu==1.8.0
openai==1.35.0
langchain==0.3.0

# Data
pandas==2.2.0
numpy==1.26.0
pypdf==5.0.0
python-dotenv==1.0.0

# Auth/Users
passlib==1.7.4
python-jose==3.3.0
python-multipart==0.0.9

# Frontend (for web prototype)
streamlit==1.35.0

### PHASE 1: KNOWLEDGE BASE

Data Collection Tasks

   - Find 50+ Montessori/respectful parenting PDFs

  -  Find 30+ articles on parent emotional regulation

  -  Find 20+ articles on childhood trauma and parenting

   - Find 10+ books on polyvagal theory for parents

   - Find real parent questions from forums (Reddit, parenting groups)

  -  Organize all files in data/raw/ by category

Data Processing Tasks

   - Write script to extract text from PDFs

   - Clean text (remove headers, footers, weird characters)

   - Split into chunks (500-1000 chars with overlap)

   - Add metadata: topic, source, intended audience

   - Create embeddings using sentence-transformers

   - Store in FAISS index

   - Test search with 20 sample queries

### Files to Create

In [ ]:
src/
  ├── data_loader.py (already have)
  ├── text_cleaner.py
  ├── chunker.py
  ├── embeddings.py (already have FAISS version)
  └── metadata_manager.py

### PHASE 2: USER SYSTEM + DATABASE 

#### Database Schema to Build sql

-- Users table
CREATE TABLE users (
    id UUID PRIMARY KEY,
    email TEXT UNIQUE,
    password_hash TEXT,
    role TEXT, -- 'parent', 'grandparent', 'nanny'
    created_at TIMESTAMP,
    onboarding_complete BOOLEAN
);

-- Parent profiles (extends users)
CREATE TABLE parent_profiles (
    user_id UUID REFERENCES users(id),
    childhood_triggers TEXT[], -- optional, self-reported
    parenting_goals TEXT[],
    therapy_status TEXT, -- 'none', 'current', 'past'
    preferred_approach TEXT -- 'montessori', 'respectful', etc
);

-- Child profiles
CREATE TABLE children (
    id UUID PRIMARY KEY,
    parent_id UUID REFERENCES users(id),
    name TEXT,
    birth_date DATE,
    temperament TEXT, -- 'easy', 'spirited', 'sensitive'
    developmental_notes TEXT,
    interests TEXT[]
);

-- Conversations
CREATE TABLE conversations (
    id UUID PRIMARY KEY,
    user_id UUID REFERENCES users(id),
    child_id UUID REFERENCES children(id), -- null if general
    started_at TIMESTAMP,
    ended_at TIMESTAMP
);

-- Messages
CREATE TABLE messages (
    id UUID PRIMARY KEY,
    conversation_id UUID REFERENCES conversations(id),
    role TEXT, -- 'user' or 'assistant'
    content TEXT,
    emotional_state JSONB, -- detected emotion
    timestamp TIMESTAMP
);

-- Emotional patterns (learned over time)
CREATE TABLE emotional_patterns (
    user_id UUID REFERENCES users(id),
    trigger_situation TEXT,
    detected_emotion TEXT,
    frequency INTEGER,
    last_occurrence TIMESTAMP
);


### Backend API Endpoints to Build
```
text

POST   /auth/register
POST   /auth/login
GET    /auth/me

POST   /profiles/parent
PUT    /profiles/parent
GET    /profiles/parent

POST   /profiles/child
GET    /profiles/child/{child_id}
PUT    /profiles/child/{child_id}
DELETE /profiles/child/{child_id}

POST   /chat
GET    /chat/history
GET    /chat/conversation/{conversation_id}

GET    /insights/patterns
GET    /insights/triggers

Files to Create
text

src/
  ├── database.py (SQLAlchemy models)
  ├── auth.py (JWT authentication)
  ├── routers/
  │   ├── auth.py
  │   ├── users.py
  │   ├── children.py
  │   ├── chat.py
  │   └── insights.py
  ├── models/
  │   ├── user.py
  │   ├── child.py
  │   └── conversation.py
  └── main.py (FastAPI app)
```



### PHASE 3: PARENT-FIRST AI ENGINE 

#### Emotional Detection System

    - Build emotion classifier (rule-based first, ML later)

    - Create response templates for each emotion state

    - Build context assembler (pulls user history + child data)

    - Create prompt engineering system

### AI Response Pipeline
python

# Pseudo-code of what to build
```

def generate_response(user_input, user_id, child_id=None):
    # 1. Get user profile and history
    user = db.get_user(user_id)
    child = db.get_child(child_id) if child_id else None
    recent_context = db.get_recent_conversations(user_id)
    
    # 2. Detect emotional state
    emotion = emotion_detector.detect(user_input)
    
    # 3. Get relevant knowledge from FAISS
    knowledge = faiss_search(user_input, top_k=5)
    
    # 4. Build personalized prompt
    prompt = prompt_builder.build(
        user=user,
        child=child,
        emotion=emotion,
        knowledge=knowledge,
        history=recent_context
    )
    
    # 5. Call LLM (OpenAI or local)
    response = llm.generate(prompt)
    
    # 6. Save to database
    db.save_message(user_id, user_input, emotion, response)
    
    # 7. Update emotional patterns
    db.update_patterns(user_id, emotion, user_input)
    
    return response
```

### Files to Create

```

src/
  ├── emotion/
  │   ├── detector.py
  │   ├── patterns.py
  │   └── templates.py
  ├── rag/
  │   ├── retriever.py
  │   ├── reranker.py
  │   └── context_builder.py
  ├── prompts/
  │   ├── builder.py
  │   ├── parent_regulation.yaml
  │   ├── child_understanding.yaml
  │   └── technique_suggestion.yaml
  └── llm/
      ├── openai_client.py
      └── response_parser.py
```
#### Phase 3 Costs:

    OpenAI API: $20-50/month during development

    OR use local Llama: $0 (but slower)



### PHASE 4: WEB PROTOTYPE (Weeks 11-12)
Streamlit Web App (Fastest Path)
python

#### app.py structure
import streamlit as st
from src.api_client import ParentAssistantAPI

st.set_page_config(page_title="ParentFirst", layout="wide")

# Auth
```
if not st.session_state.get("logged_in"):
    login_page()
else:
    # Sidebar with profiles
    with st.sidebar:
        show_user_profile()
        show_children_selector()
        show_recent_conversations()
    
    # Main chat area
    chat_interface()
    
    # Insights dashboard
    with st.expander("Your Emotional Patterns"):
        show_patterns_chart()
```

### Features to Build

-    Login/signup screen

 -   Parent profile setup (childhood questions, goals)

  -  Add child profile (age, temperament)

   - Chat interface with message history

    - Emotion visualization dashboard

-    Settings panel

#### Deployment

 -   Push code to GitHub

  -  Connect to Streamlit Cloud (free)

   - Set up environment variables

    - Deploy and test

#### Phase 4 Costs:

-    Streamlit Cloud: Free

 -   Domain name (optional): $15/year



### PHASE 5: MOBILE APP 
#### Option A: React Native 

Programs to Install
Program	Purpose	Cost
Node.js	JavaScript runtime	Free
npm/yarn	Package manager	Free
React Native CLI	Mobile framework	Free
Android Studio	Android emulator	Free
Xcode	iOS emulator (Mac only)	Free

Accounts Needed
Account	Purpose	Cost
Apple Developer	iOS app store	$99/year
Google Play Console	Android app store	$25 one-time
Mobile App Features to Build
text

```
├── Setup React Native project
├── Build login/signup screens
├── Build profile setup wizard
└── Build chat interface


├── Add push notifications
├── Add voice input
├── Add offline storage
└── Add child profile management


├── Add emotional patterns dashboard
├── Add settings and preferences
├── Test on both platforms
└── Prepare app store listings
```


#### Files to Create

```
mobile-app/
├── src/
│   ├── screens/
│   │   ├── Login.js
│   │   ├── Signup.js
│   │   ├── Onboarding.js
│   │   ├── Chat.js
│   │   ├── Profile.js
│   │   └── Insights.js
│   ├── components/
│   │   ├── MessageBubble.js
│   │   ├── ChildSelector.js
│   │   └── EmotionChart.js
│   ├── api/
│   │   └── client.js
│   └── utils/
│       ├── auth.js
│       └── storage.js
├── App.js
└── package.json
```

#### Phase 5 Costs:

-    Apple Developer: $99/year

   - Google Play: $25 one-time

    - Total first year: $124



### PHASE 6: SUBSCRIPTION SYSTEM 
#### Payment Processing

__Account Needed:__ Stripe (free to start, takes % of transactions)
__Subscription Tiers to Build__
Tier	 Price	    Features
Free	 $0	        1 parent, 1 child, basic chat
Premium	 $8/mo	    2 parents, 3 children, full memory, patterns
Family	 $15/mo   	Add grandparents, family insights
Pro	     $25/mo	    Nanny access, professional tools

Code to Add

-    Stripe webhook integration

 -   Subscription status checks in API

  -  Feature flags based on tier

-    Upgrade/downgrade flows

 -   Payment history

Phase 6 Costs:

    Stripe: 2.9% + $0.30 per transaction

    No monthly fees





### PHASE 7: ADVANCED AI LEARNING (Months 8-12)
#### Data Collection for Fine-tuning

    - Collect 5000+ real parent conversations

    - Anonymize all data

    - Create training pairs (user input → ideal response)

 -   Validate with parenting experts

Fine-tuning Pipeline

  -  Set up GPU environment (Google Colab or local)

 -   Fine-tune Llama 3.1 8B on your data

  -  Evaluate against test set

  -  Deploy fine-tuned model

  -  A/B test vs. base model

Phase 7 Costs:

   - Google Colab Pro: $10/month

   - OR university GPU: Free

   - OR cloud GPU: $50-100/month,m



#### PHASE 8: PRODUCTION DEPLOYMENT 
### Production Infrastructure
```
Service	Purpose	Monthly Cost
PostgreSQL (Render)	Main database	    $7-15
Redis (Upstash)	Caching/sessions	    $0-10
Backend (Railway)	API server	    $5-20
File storage (AWS S3)	User uploads	    $0-5
Monitoring (Sentry)	Error tracking	    $0-29
Analytics (Mixpanel)	User behavior	    $0-25
Total		                            $12-104

```
Legal/Compliance

*    Terms of service (template: $0)

 *   Privacy policy (template: $0)

 *   COPPA compliance (if under 13)

  *  GDPR compliance (if EU users)

 *   Consult lawyer (optional: $500-2000)

### COMPLETE COST SUMMARY
```
Minimum Viable Product (Months 1-4)
Item	                        Cost
Development tools	            
OpenAI API (4 months)	       $80-200
Hosting (free tiers)	       $0
Domain (optional)	       $15

Total MVP	               $95-215


Full Production (Year 1)
Item	                        Cost
MVP costs	                $95-215
Apple Developer	                $99
Google Play	                $25
Production hosting (avg $30/mo)	$360
OpenAI API         	        $200-600
Stripe fees 
(depends on revenue)	         2.9% + $0.30/transaction

Total Year 1	               $779-1299

```